# Parallelization

This pattern enables additional efficiencies by running multiple sub-tasks in parallel and then collecting results for synthesis. Use cases include: deep research, travel planning, customer feedback analysis, etc.

## LangChain vs Flyte v2 — Key Differences

In the context of this pattern, these are the main differences between Langchaing and Flyte V2:


| Aspect | LangChain | Flyte v2 |
|---------|-----------|----------|
| **Parallel execution** | `RunnableParallel` (LCEL) | `asyncio.gather` + Flyte tasks |
| **LLM calls** | `ChatOpenAI` via LCEL | Direct `AsyncOpenAI` client |
| **Caching** | None | `cache="auto"` per task |
| **Execution** | In-process only | Local or remote (containers) |
| **Dependencies** | langchain-openai, langchain-core | flyte, openai |


Below is a concise diff showing how the original LangChain example maps to the Flyte v2 + native Python implementation:

```diff
--- LangChain (original)
+++ Flyte v2 + native asyncio

- import os
- import asyncio
- from langchain_openai import ChatOpenAI
- from langchain_core.prompts import ChatPromptTemplate
- from langchain_core.output_parsers import StrOutputParser
- from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough
+ import os
+ import asyncio
+ import flyte
+ from flyte import TaskEnvironment, Resources, Secret
+ from openai import AsyncOpenAI

- llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
- summarize_chain = ChatPromptTemplate.from_messages([...]) | llm | StrOutputParser()
- questions_chain = ChatPromptTemplate.from_messages([...]) | llm | StrOutputParser()
- terms_chain = ChatPromptTemplate.from_messages([...]) | llm | StrOutputParser()
+ env = TaskEnvironment(name="parallel_env", ...)
+ client = AsyncOpenAI()
+ async def _summarize(topic: str) -> str: ...  # Direct OpenAI call
+ async def _questions(topic: str) -> str: ...
+ async def _terms(topic: str) -> str: ...

- map_chain = RunnableParallel({
-     "summary": summarize_chain,
-     "questions": questions_chain,
-     "key_terms": terms_chain,
-     "topic": RunnablePassthrough(),
- })
- full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()
+ @env.task
+ async def summarize_task(topic: str) -> str: ...
+ @env.task
+ async def questions_task(topic: str) -> str: ...
+ @env.task
+ async def terms_task(topic: str) -> str: ...

- response = await full_parallel_chain.ainvoke(topic)
+ summary, questions, terms = await asyncio.gather(
+     summarize_task(topic), questions_task(topic), terms_task(topic)
+ )
+ response = await synthesize_task(topic, summary, questions, terms)
```


## Implementation with Flyte v2

This notebook refactors the LangChain `RunnableParallel` example to use Flyte v2 and the native OpenAI client. The three parallel sub-tasks (summarize, questions, key terms) run as Flyte tasks that can execute in parallel containers when run remotely.

1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' openai

2. Export your API key

In [ ]:
%env OPENAI_API_KEY=<your-api-key>

3. Import dependencies and configure the Flyte TaskEnvironment

In [ ]:
import os
import asyncio
import flyte
from flyte import TaskEnvironment, Resources, Secret
from openai import AsyncOpenAI


api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not set. Use %env OPENAI_API_KEY=... and re-run this cell."
    )
client = AsyncOpenAI(api_key=api_key)


env = TaskEnvironment(
    name="parallel_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    image=flyte.Image.from_debian_base().with_pip_packages("openai"),
)

4. Define the three parallel sub-tasks and the synthesis task

Each sub-task is a Flyte task that calls OpenAI directly. When run remotely, Flyte schedules them in parallel containers.

In [ ]:
async def _llm_call(system: str, user: str) -> str:
    """Helper: single LLM call with system + user messages."""
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return (resp.choices[0].message.content or "").strip()


@env.task
async def summarize_task(topic: str) -> str:
    """Summarize the topic concisely."""
    return await _llm_call(
        "Summarize the following topic concisely:",
        topic,
    )


@env.task
async def questions_task(topic: str) -> str:
    """Generate three interesting questions about the topic."""
    return await _llm_call(
        "Generate three interesting questions about the following topic:",
        topic,
    )


@env.task
async def terms_task(topic: str) -> str:
    """Identify 5-10 key terms from the topic, comma-separated."""
    return await _llm_call(
        "Identify 5-10 key terms from the following topic, separated by commas:",
        topic,
    )


@env.task
async def synthesize_task(
    topic: str, summary: str, questions: str, key_terms: str
) -> str:
    """Synthesize a comprehensive answer from the parallel results."""
    system = """Based on the following information:
Summary: {summary}
Related Questions: {questions}
Key Terms: {key_terms}
Synthesize a comprehensive answer."""
    return await _llm_call(
        system.format(summary=summary, questions=questions, key_terms=key_terms),
        f"Original topic: {topic}",
    )

5. Orchestrate the parallel workflow

The main task runs the three sub-tasks in parallel with `asyncio.gather`. Flyte schedules them concurrently (locally or in separate containers when remote).

In [ ]:
@env.task
async def parallel_research(topic: str) -> str:
    """Run summarize, questions, and terms in parallel, then synthesize."""
    summary, questions, key_terms = await asyncio.gather(
        summarize_task(topic),
        questions_task(topic),
        terms_task(topic),
    )
    return await synthesize_task(topic, summary, questions, key_terms)

6. Run locally

In [ ]:
# Local execution (same pattern as routing.ipynb)
if "OPENAI_API_KEY" not in os.environ:
    print("Set OPENAI_API_KEY in your environment before running this demo.")
else:
    run = flyte.with_runcontext(mode="local").run(
        parallel_research, topic="The history of space exploration"
    )
    run.wait()
    print(run.outputs()[0])

### Remote execution

Configure `flyte.init()` with your cluster endpoint, then:

```bash
flyte run parallelization.ipynb parallel_research --topic "The history of space exploration"
```